# Citus Database Setup and SDK Demo (Single Node)

This notebook demonstrates how to set up a Citus database using a single-node docker container, populate the patch table with dummy data, and implement a Python SDK for interacting with the database. Worker node logic is omitted for single-node setup.

## 1. Install and Import Required Libraries

Install `psycopg` if not already installed, and import all required libraries for database interaction.

In [ ]:
# Install psycopg if needed (uncomment if running in a new environment)
# !pip install psycopg[binary]

import psycopg
from psycopg.rows import dict_row
import random
import datetime
import base64
from db_client import CitusHeadClient
import os

In [ ]:
# Import DB connection constants from constants.py
from constants import (
    CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD
)

In [ ]:
# Set DB connection variables from constants (single-node)
DB_HOST = CITUS_HEAD_HOST
DB_PORT = CITUS_HEAD_PORT
DB_NAME = CITUS_HEAD_DB
DB_USER = CITUS_HEAD_USER
DB_PASSWORD = CITUS_HEAD_PASSWORD

In [ ]:
NUM_PATCHES = 1000

## 0. Drop All Tables (Clean Start)

Drop all tables if they exist to ensure a clean setup.

In [ ]:
head_client = CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD)
head_client.drop_all_tables()
print("All tables dropped (if existed).")


## 2. Connect to Citus Node

Establish a connection to the Citus/Postgres node using psycopg. Store connection parameters securely (e.g., using environment variables).

In [ ]:
# Use CitusHeadClient for connection
def get_head_connection():
    return CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD).get_connection()

# Test connection
with get_head_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT version();')
        print('Connected to:', cur.fetchone()['version'])

## 3. Create Database Schema (Tables)

Create all tables as described in the technical design document, including distributed and reference tables. Use Citus distribution commands where required.

In [ ]:
head_client.setup_schema()
print("Schema and distribution setup complete.")


## 4. Verify Table Creation

Query the information schema to verify that all tables have been created successfully.

In [ ]:
# List all tables in the public schema
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        tables = [row['table_name'] for row in cur.fetchall()]
        print('Tables in public schema:', tables)

## 5. Insert Dummy Data into Patch Table

Generate and insert dummy data into the patch table, ensuring all required fields are populated and constraints are respected.

In [ ]:
# Helper: Insert dummy project, image, label_class for FK constraints
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO project (project_name, description) VALUES (%s, %s) RETURNING project_id;", ('Demo Project', 'For dummy data'))
        project_id = cur.fetchone()['project_id']
        cur.execute("INSERT INTO image (project_id, name, image_path, upload_ts, base_mag, base_width, base_height, deepzoom_tilesize) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) RETURNING image_id;",
                    (project_id, 'Demo Image', '/tmp/demo.tif', datetime.datetime.now(), 20.0, 10000, 8000, 256))
        image_id = cur.fetchone()['image_id']
        cur.execute("INSERT INTO label_class (project_id, name, color_code, event_ts) VALUES (%s, %s, %s, %s) RETURNING label_class_id;",
                    (project_id, 'Tumor', '#FF0000', datetime.datetime.now()))
        label_class_id = cur.fetchone()['label_class_id']
        print(f"Inserted project_id={project_id}, image_id={image_id}, label_class_id={label_class_id}")

def random_bytes(size=128):
    return os.urandom(size)

for i in range(NUM_PATCHES):
    patch_id = head_client.insert_patch(1000 + i, label_class_id, image_id, 20.0, random_bytes())
    print(f"Inserted patch_id={patch_id}")

In [ ]:
# Check number of shards for the patch table and print row counts per shard, including empty shards
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        # Number of shards
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'patch' table: {num_shards}")
        # Row counts per shard, including empty
        cur.execute("""
            SELECT s.shardid, COALESCE(count(p.patch_id), 0) as row_count
            FROM pg_dist_shard s
            LEFT JOIN patch p ON get_shard_id_for_distribution_column('patch', p.patch_id) = s.shardid
            WHERE s.logicalrelid = 'patch'::regclass
            GROUP BY s.shardid
            ORDER BY s.shardid;
        """)
        rows = cur.fetchall()
        empty_count = 0
        for row in rows:
            print(f"Shard {row['shardid']}: {row['row_count']} rows")
            if row['row_count'] == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")

## 6. Verify Dummy Data in Patch Table

Query the patch table to confirm that dummy data has been inserted correctly.

In [ ]:
# Query and display dummy patch data
for row in head_client.fetch_patches(limit=10):
    print(row)

## 7. Implement db_client SDK: Head Node Level

Write Python classes and functions in `db_client.py` to interact with the database at the Citus head node level, including connection management and basic CRUD operations.

In [ ]:
# db_client.py will be implemented in the next step.
# Example usage for SDK will be shown after SDK implementation.

In [ ]:
# Example: Using db_client SDK (single-node)
# Uses constants.py for all connection parameters
head_client = CitusHeadClient()
print('Patches:', head_client.fetch_patches(limit=3))

# Insert a new patch (dummy data)
# patch_id = head_client.insert_patch(2000, 1, 1, 20.0, b'dummybytes')
# print('Inserted patch_id:', patch_id)

<!-- Worker node logic omitted for single-node setup -->